# Env39 — Integrated Street-Morphology Results

This is the principal professor-facing Env39 results notebook. It integrates accepted Phase 7F and Phase 7G artifacts without reimplementing or changing their metrics.

## 1. Research objective

Establish one concise, reproducible Env39 baseline and a long-form schema that a later Env38 analysis can populate identically. The purpose is controlled comparison: topology is held fixed while geometric realization may vary.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
ROOT = next((p for p in candidates if (p / 'src' / 'geogami_morphology').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from the repository root or notebooks directory.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from IPython.display import Markdown, SVG, display
from geogami_morphology.integrated import integrate_results

result = integrate_results('env39', canonical_run='latest', project_root=ROOT, publish=False)
metrics = pd.DataFrame(result.metrics)
core = pd.DataFrame(result.core_metrics)
assert len(core) == 19 and set(metrics['metric_tier']) == {'core', 'supplementary'}
print('INTEGRATED_SOURCE_LOADING: PASS')

## 2. Selected canonical run and provenance

The selected run is resolved through the verified Phase 7C pointer. Phase 7H checks the canonical file SHA-256, scientific-content signature, topology signature, Phase 7D graph manifest, Phase 7F topology manifest, Phase 7G geometry manifest, and current metric-configuration hash before presenting results.

In [ ]:
provenance = {
    'environment': result.selection.environment,
    'canonical_run_id': result.selection.run_id,
    'canonical_sha256': result.selection.file_sha256,
    'scientific_content_signature': result.selection.scientific_content_signature,
    'topology_signature': result.selection.topology_signature,
}
display(pd.Series(provenance, name='value').to_frame())
print('INTEGRATED_PROVENANCE_CHAIN: PASS')

## 3. Analysis architecture

The chain of custody is: canonical GeoPackage → NetworkX/OSMnx graph adapter → Phase 7F topology and weighted-network results → Phase 7G planar geometry and orientation results → Phase 7H classification and presentation. Integration reads stored summaries; it does not calculate scientific metrics.

## 4. Metric-family classification

- **topology_controlled:** connectivity-only values expected to remain identical when the frozen topology is maintained.
- **geometry_weighted_network:** controlled graph calculations using physical edge length as cost.
- **geometric_morphology:** values derived directly from planar street geometry.
- **orientation_morphology:** axial-direction values under the fixed Phase 7G binning and weighting definitions.

The `comparison_role` column states whether a value is a topology control, geometry-sensitive result, orientation-sensitive result, or contextual method parameter.

In [ ]:
display(metrics.groupby(['metric_family', 'comparison_role', 'metric_tier']).size().rename('metric_count').to_frame())
print('METRIC_FAMILY_CLASSIFICATION: PASS')

## 5. Topology-control results

These values depend only on graph connectivity. Counts have count units, proportions are dimensionless, and hop distances are unweighted. In a valid later Env38 comparison, this family should be identical and will function as the experimental control check.

In [ ]:
columns = ['metric_label', 'value', 'units', 'population', 'weighting', 'comparison_role']
display(metrics.loc[metrics.metric_family == 'topology_controlled', columns].reset_index(drop=True))
print('TOPOLOGY_CONTROL_RESULTS: PASS')

## 6. Geometry-weighted network results

Mean weighted shortest-path length uses canonical physical edge geometry length as routing cost. Its units are local units, not metres. It may change for Env38 even when connectivity is fixed because changed edge lengths alter least-cost paths.

In [ ]:
display(metrics.loc[metrics.metric_family == 'geometry_weighted_network', columns].reset_index(drop=True))
print('GEOMETRY_WEIGHTED_NETWORK_RESULTS: PASS')

## 7. Geometric morphology results

Lengths are planar GeoGami local units. Circuity is dimensionless: values near one indicate that realized LineStrings are close to their endpoint chords. These results can vary when geometry changes on the controlled topology.

In [ ]:
display(metrics.loc[metrics.metric_family == 'geometric_morphology', columns].reset_index(drop=True))
print('GEOMETRIC_MORPHOLOGY_RESULTS: PASS')

## 8. Orientation morphology results

Entropy is reported in nats and as a normalized dimensionless value. Lower entropy means greater directional concentration under the configured 36 five-degree axial bins; it is not a quality ranking. Fourfold order `phi` ranges from zero to one, with higher values indicating stronger fourfold axial alignment under the stated observation weighting.

In [ ]:
display(metrics.loc[metrics.metric_family == 'orientation_morphology', columns].reset_index(drop=True))
print('ORIENTATION_MORPHOLOGY_RESULTS: PASS')

## 9. Core metric table

The 19 core metrics form the main research table: eight topology controls, one geometry-weighted routing result, four direct geometry results, and six orientation results. This compact set describes control validity and principal morphology without promoting every detailed node or edge statistic into the headline analysis.

In [ ]:
display(core[['metric_id', 'metric_label', 'metric_family', 'value', 'units', 'population', 'weighting', 'comparison_role']])
print('CORE_METRIC_TABLE: 19 METRICS | PASS')

## 10. Selected supplementary results

Supplementary metrics provide connectivity, distribution, and method context. Individual node and edge centralities remain in the detailed Phase 7F CSV tables and are intentionally not duplicated here.

In [ ]:
supplementary = metrics.loc[metrics.metric_tier == 'supplementary']
display(supplementary[['metric_id', 'metric_label', 'metric_family', 'value', 'units', 'comparison_role']].reset_index(drop=True))
print('SUPPLEMENTARY_RESULTS: PASS')

## 11. Key scientific figures

The integrated set is intentionally limited to five headless-safe SVGs. The network and degree views are deterministic presentation views; the circuity and orientation plots reuse the accepted Phase 7G figures so no competing plotting interpretation is introduced.

In [ ]:
for number, figure in enumerate(result.figure_catalog, start=1):
    path = ROOT / figure['path']
    assert path.is_file()
    display(Markdown(f"**Figure {number}. {figure['label']}**  \n{figure['provenance']}"))
    display(SVG(filename=str(path)))
print('INTEGRATED_FIGURES: 5 FOUND | PASS')

## 12. What is controlled versus what can vary

Identical topology metrics will be evidence that experimental topology control was maintained. Geometric and orientation metrics may differ because the realized street shapes may differ. Geometry-weighted network metrics may differ because physical lengths change routing costs on otherwise fixed connectivity. This design supports attribution within the controlled setup, not an unrestricted causal claim.

## 13. Interpretation of grid-like Env39

Env39 is one connected physical graph with a substantial cycle rank and a degree composition containing many three- and four-way junctions. Circuity is low and axial orientation is concentrated, while both chord and segment observations show strong fourfold order. These are descriptive properties, not evidence that Env39 is better or that it causes improved navigation.

## 14. Historical methodology note

Earlier approximate project outputs reported orientation entropy around 1.5144 and `phi` around 0.9966. They are retained only as historical outputs and are not directly comparable: their observation population, weighting, binning, and/or formula documentation was not sufficiently aligned with the accepted method. Phase 7G now provides separate explicit chord and segment-length-weighted baselines and is the current analytical method.

## 15. Export package

The integrated directory contains the all-metric comparison CSV, compact core CSV/JSON, methodology JSON, manifest, one-page Markdown summary, and two integrated SVG views. Phase 7G figures are referenced rather than copied.

In [ ]:
artifacts = result.manifest['artifacts']
artifact_table = pd.DataFrame([{'artifact': name, **record} for name, record in artifacts.items()])
assert all((ROOT / p).is_file() for p in artifact_table['path'])
display(artifact_table)
print('INTEGRATED_EXPORT_PACKAGE: PASS')

## 16. Reproducibility summary

Every integrated output is hash-recorded. The manifest also records all canonical identities, upstream manifest hashes, metric-configuration hash, Git commit and dirty state, software versions, schema version, and pre/post source-artifact hashes. The notebook loads the package read-only.

In [ ]:
assert result.manifest['comparison_schema_version'] == '1.0.0'
assert len(result.manifest['provenance_chain']) == 4
print('INTEGRATED_REPRODUCIBILITY: PASS')

## 17. How the same workflow will later analyze Env38

After an authorized Env38 canonical publication, the same Phase 7D → 7F → 7G → 7H workflow can populate identical columns and stable `metric_id` values with `environment=env38`. No Env38 values are fabricated here. Comparison begins by confirming topology-control equality, then evaluates geometry-weighted, direct geometry, and orientation differences under their documented units and populations.

## 18. Final Env39 baseline summary

Env39 now has a concise, provenance-linked baseline ready to serve as the control-side analytical contract for a later Env38 comparison. Interpretation remains bounded by the local-coordinate definitions and the controlled experimental design.

In [ ]:
display(Markdown((result.output_dir / 'env39_baseline_summary.md').read_text(encoding='utf-8')))
print('NOTEBOOK_04_EXECUTION: PASS')